In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io

# Markdown table data
markdown_data = """
| timestamp | filename | concurrency | requests | successful | failed | timeouts | throughput_req_per_sec | latency_avg_ms | latency_p95_ms | load_before | load_after | memory_used_before_mb | memory_used_after_mb |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-06-11T22:20:35Z | t01_workers4_concurrency50_allmodels.json | 50 | 50 | 50 | 0 | 0 | 0.42 | 81433.42 | 118343.04 | 0.16 | 4.7 | 2087 | 2411 |
| 2026-06-11T22:47:31Z | t02_workers4_concurrency30_allmodels.json | 30 | 50 | 50 | 0 | 0 | 0.5 | 48161.44 | 61663.03 | 0.09 | 9.25 | 2393 | 2481 |
| 2026-06-15T18:17:26Z | t03_workers4_concurrency30_allmodels_run01.json | 30 | 50 | 50 | 0 | 0 | 0.5 | 48081.18 | 61107.53 | 0.06 | 8.63 | 2646 | 2711 |
| 2026-06-15T18:24:41Z | t04_workers4_concurrency30_allmodels_run02.json | 30 | 50 | 50 | 0 | 0 | 0.48 | 50237.62 | 65981.89 | 0.29 | 9.46 | 2287 | 2736 |
| 2026-06-15T18:38:39Z | t05_workers4_concurrency30_allmodels_run03.json | 30 | 50 | 50 | 0 | 0 | 0.51 | 51639.67 | 66392.62 | 0.25 | 11.37 | 2305 | 2748 |
| 2026-06-15T18:56:53Z | t06_workers4_concurrency30_allmodels_run04.json | 30 | 50 | 50 | 0 | 0 | 0.48 | 50744.5 | 65083.41 | 0.4 | 9.41 | 2347 | 2752 |
| 2026-06-15T19:24:19Z | t07_workers4_concurrency30_allmodels_run05.json | 30 | 50 | 50 | 0 | 0 | 0.49 | 50364.05 | 62667.82 | 0.57 | 8.15 | 2339 | 2786 |
| 2026-06-15T19:28:40Z | t08_workers4_concurrency30_allmodels_run06.json | 30 | 50 | 50 | 0 | 0 | 0.51 | 52247.12 | 67044.5 | 1.26 | 12.23 | 2382 | 2802 |
| 2026-06-15T19:32:03Z | t09_workers4_concurrency30_allmodels_run07.json | 30 | 50 | 50 | 0 | 0 | 0.44 | 57301.41 | 104705.72 | 2.87 | 6.76 | 2365 | 2805 |
| 2026-06-18T17:45:43Z | t10_workers4_concurrency30_svm_firstattempt.json | 30 | 50 | 6 | 44 | 44 | 0.01 | 224644.46 | 237323.04 | 0.42 | 13.16 | 2202 | 2693 |
| 2026-06-19T16:08:52Z | t11_workers4_threads100_concurrency30_svm.json | 30 | 50 | 4 | 46 | 46 | 0.01 | 173618.77 | 179038.93 | 0 | - | 2011 | - |
| 2026-06-19T16:28:59Z | t12_workers4_threads50_concurrency30_svm.json | 30 | 50 | 41 | 9 | 9 | 0.05 | 337488.6 | 404241.22 | 0.1 | 6.65 | 2466 | 2498 |
| 2026-06-19T18:16:55Z | t13_workers4_threads40_concurrency30_svm.json | 30 | 50 | 2 | 48 | 48 | 0 | 276368.77 | 298086.98 | 0.67 | 19.5 | 2398 | 2490 |
| 2026-07-04T14:48:21Z | t14_workers4_concurrency30_allmodels_timeout01.json | 30 | 50 | 0 | 50 | 50 | 0 | - | - | 1.29 | - | 2018 | - |
| 2026-07-04T14:48:40Z | t15_workers4_concurrency30_allmodels_timeout02.json | 30 | 50 | 0 | 50 | 50 | 0 | - | - | - | - | - | - |
| 2026-07-04T16:06:55Z | t16_workers4_concurrency15_allmodels.json | 15 | 50 | 50 | 0 | 0 | 0.07 | 202147.02 | 256146.07 | 0.62 | 7.18 | 2005 | 2353 |
"""

# Read the markdown table into a pandas DataFrame
df = pd.read_csv(io.StringIO(markdown_data), sep='|', skipinitialspace=True)

# Clean up column names (remove leading/trailing spaces)
df.columns = df.columns.str.strip()

# Drop the first and last empty columns created by the leading/trailing pipes
df = df.iloc[:, 1:-1]

# Replace '-' with NaN and convert numeric columns to float
df = df.replace('-', pd.NA)

numeric_cols = [
    'concurrency', 'requests', 'successful', 'failed', 'timeouts', 
    'throughput_req_per_sec', 'latency_avg_ms', 'latency_p95_ms', 
    'load_before', 'load_after', 'memory_used_before_mb', 'memory_used_after_mb'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Shorten filename for better plotting
df['short_name'] = df['filename'].str.replace('.json', '').str.replace('workers4_', '')

# Set up the plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)
plt.rcParams['xtick.labelsize'] = 8

# Create a figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Stress Test Results Analysis', fontsize=18, fontweight='bold')

# 1. Throughput vs Test Case
sns.barplot(data=df, x='short_name', y='throughput_req_per_sec', ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Throughput (req/sec)', fontsize=14)
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45, ha='right')
axes[0, 0].set_ylabel('Requests per Second')

# 2. Latency (Avg and P95) vs Test Case
df_melted_latency = df[['short_name', 'latency_avg_ms', 'latency_p95_ms']].melt(id_vars='short_name', var_name='Metric', value_name='Latency (ms)')
sns.barplot(data=df_melted_latency, x='short_name', y='Latency (ms)', hue='Metric', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Latency (Avg vs P95)', fontsize=14)
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45, ha='right')
axes[0, 1].legend(title='Metric')

# 3. Success vs Failed vs Timeouts
df_melted_status = df[['short_name', 'successful', 'failed', 'timeouts']].melt(id_vars='short_name', var_name='Status', value_name='Count')
sns.barplot(data=df_melted_status, x='short_name', y='Count', hue='Status', ax=axes[1, 0], palette='Set1')
axes[1, 0].set_title('Request Status (Successful, Failed, Timeouts)', fontsize=14)
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45, ha='right')
axes[1, 0].legend(title='Status')

# 4. Memory Usage (Before vs After)
df_melted_memory = df[['short_name', 'memory_used_before_mb', 'memory_used_after_mb']].melt(id_vars='short_name', var_name='State', value_name='Memory (MB)')
sns.barplot(data=df_melted_memory, x='short_name', y='Memory (MB)', hue='State', ax=axes[1, 1], palette='coolwarm')
axes[1, 1].set_title('Memory Usage (Before vs After)', fontsize=14)
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')
axes[1, 1].legend(title='State')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('stress_test_results.png', dpi=300, bbox_inches='tight')
print("Plot saved successfully as 'stress_test_results.png'")